#IMPLEMENTATION


In [ ]:
!pip install langchain
!pip install langchain_text_splitters
!pip install openai
!pip install -U "langchain[google-genai]"
!pip install pydantic
!pip install langchain-groq
!pip install neo4j
!pip install networkx
!pip install pyvis
!pip install cdlib
!pip install leidenalg

In [3]:
def chunked(text):
  splitter = RecursiveCharacterTextSplitter(
      chunk_size=1000,
      chunk_overlap=200,
      separators=["\n\n", "\n", " ", ""])
  chunks=splitter.split_text(text)
  return chunks
EntityType = Literal[
    # CLASS A: BIOLOGICAL AGENTS
    "PATHOGEN",          # Viruses, Bacteria, Fungi (e.g., H3N2, E. coli)
    "HOST",              # Patients, Animals, Demographics (e.g., Refugee patients, Children, Ferrets)

    # CLASS B: MOLECULAR REALITY (Crucial for your data)
    "GENETIC_ENTITY",    # Genes, RNA, DNA, Primers, Amplicons (e.g., HA Gene, PCR Primers)
    "MOLECULAR_VARIANT", # Mutations, Substitutions, Strains, Isolates (e.g., K145N, Nepal Isolates)
    "CHEMICAL",          # Drugs, Proteins, Reagents, Antibodies (e.g., Lisinopril, HA Protein)

    # CLASS C: CLINICAL REALITY
    "CONDITION",         # Diseases, Symptoms, Syndromes (e.g., Influenza, Fever)
    "PHENOMENON",        # Biological processes (e.g., Genetic Drift, Antigenic Variation)

    # CLASS D: INTERVENTION & CONTEXT
    "PROCEDURE",         # Tests, Assays, Sequencing, Vaccinations (e.g., PCR, HI Assay)
    "DEVICE",            # Machines, Kits (e.g., Magnapure LX)
    "LOCATION",          # Geographic or Anatomical (e.g., Nepal, Throat)
    "METRIC",            # Measurements, Titers (e.g., 1:320, 4-fold increase)
    "CELL",
    "ANATOMY",          # Added these 2
    "OTHER"              # Strict fallback
]

RelationType = Literal[
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION (H3N2 causes Influenza)
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION (Tamiflu treats Influenza)
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION (Vaccine prevents Flu)

    "EXHIBITS",          # PATHOGEN -> VARIANT (H3N2 exhibits K145N mutation)
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY (PCR detects RNA)
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN (Primers target HA Gene)

    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST (Nepal Isolates originate from Nepal)
    "DIFFERS_FROM",      # VARIANT -> VARIANT (Strain A differs from Strain B)

    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME (General linker)
    "ASSOCIATED_WITH",    # Fallback for weak correlations
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
]

class Entity(BaseModel):
    name: str = Field(description="The precise scientific name. For 'Nepal Isolates', extract 'Isolates' as the name and 'Nepal' as a separate Location node.")
    type: EntityType = Field(description="The strict scientific category.If unsure of the category, STRICTLY use 'OTHER'.")

class Relation(BaseModel):
    source_entity: Entity = Field(description="The subject of the relation")
    relation_type: RelationType = Field(description="The strict scientific verb connecting them.If the relationship is complex, you MUST select 'ASSOCIATED_WITH'.")
    target_entity: Entity = Field(description="The object of the relation")
    evidence: str = Field(description="The EXACT quote from the text supporting this.")
    confidence: float = Field(description="Confidence score (0.0-1.0).")

class MedicalGraph(BaseModel):
   reasoning_trace: str = Field(
        description='''
            "THINKING STEP-BY-STEP LOGIC:\n"
            "Carefully think before answering"
            "DO NOT CREATE INFORMATION , USE INFORMATION PROVIDED IN THE TEXT ONLY."
            ""CONNECTIVITY PLANNING:\n"
            "1. IDENTIFY THE HUB: Which entity is the main subject? (e.g., 'The Patient' or 'The Virus').\n"
            "2. LINK ORPHANS: Look for entities that might become isolated. explicitly plan how to link them back to the Hub.\n"
            '''

    )
   relations: List[Relation]


system_prompt = """You are an expert Medical Knowledge Graph Engineer.
Your goal is to extract a highly precise, scientifically accurate Knowledge Graph from the provided text.

### CORE INSTRUCTION: DECONSTRUCTION OVER LABELING
Medical text often compresses complex relationships into single noun phrases.
Do NOT label complex phrases as single nodes. Instead, DECONSTRUCT them into atomic components.

Examples of Deconstruction:
- "Metastatic Lung Cancer" -> "Lung Cancer" (CONDITION) + "Metastasis" (PHENOMENON).
- "Nepal Isolates" -> "Isolates" (MOLECULAR_VARIANT) + "Nepal" (LOCATION).
- "PCR Primers" -> "Primers" (GENETIC_ENTITY) + "PCR" (PROCEDURE).

### STRICT ENTITY GUIDELINES
1. PATHOGEN: Only the active infectious agent (e.g., "H3N2", "S. aureus"). Do not confuse with the Disease.
2. CONDITION: The passive disease or symptom (e.g., "Influenza", "Pneumonia").
3. CHEMICAL: Includes drugs, proteins, and lab reagents.
4. MOLECULAR_VARIANT: Use this for specific strains, mutations, or receptor statuses (e.g., "Delta Variant", "HER2+").
5. GENETIC_ENTITY: Use this for Genes, RNA, DNA, and Primers.
6. LOCATION: Use this for both Geography (Countries) and Anatomy (Body parts).

### RELATIONSHIP USAGE RULES
- Use 'EXHIBITS' to link a Subject to its Properties or Variants (e.g., Virus -> EXHIBITS -> Mutation).
- Use 'TARGETS' for Mechanism of Action (e.g., Drug -> TARGETS -> Protein).
- Use 'ORIGINATES_FROM' to link samples/variants to their source (e.g., Tumor -> ORIGINATES_FROM -> Tissue).
- Use 'HAS_CONTEXT' to link Entities to Metrics, Time, or Locations (e.g., Patient -> HAS_CONTEXT -> Age 60).
- Use 'DETECTS' for Diagnostic tools finding a Condition or Pathogen.

### OUTPUT LOGIC
- If the text mentions "Resistance to Tamiflu", extract:
  (Tamiflu: CHEMICAL) -[TREATS]-> (Influenza: CONDITION)
  (Influenza: CONDITION) -[EXHIBITS]-> (Resistance: PHENOMENON)
  (Resistance: PHENOMENON) -[ASSOCIATED_WITH]-> (Tamiflu: CHEMICAL)

Analyze the text deeply. Bias towards granular, scientifically precise nodes.
"""

few_shot_examples = """
Input: "Patient diagnosed with hypertension. Prescribed Lisinopril 10mg."
Output: [
  {"source": "Lisinopril", "relation": "TREATS", "target": "Hypertension", "evidence": "Prescribed Lisinopril 10mg"}
]

Input: "Patient denies chest pain but reports shortness of breath."
Output: [
  {"source": "Patient", "relation": "ASSOCIATED_WITH", "target": "Shortness of Breath", "evidence": "reports shortness of breath"}
  // Note: We deliberately ignored Chest Pain because it was denied.
]
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Here are some examples of correct extraction: {examples}"),
    ("human", "Analyze this clinical note: {text}")
])


merge_query="""
MERGE (e:Entity{name:$name1})
ON CREATE
  SET e.type=$type1

MERGE (f:Entity{name:$name2})
ON CREATE
  SET f.type=$type2

MERGE (e)-[r:RELATIONSHIP]->(f)
"""
relationships=[
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION (H3N2 causes Influenza)
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION (Tamiflu treats Influenza)
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION (Vaccine prevents Flu)

    "EXHIBITS",          # PATHOGEN -> VARIANT (H3N2 exhibits K145N mutation)
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY (PCR detects RNA)
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN (Primers target HA Gene)

    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST (Nepal Isolates originate from Nepal)
    "DIFFERS_FROM",      # VARIANT -> VARIANT (Strain A differs from Strain B)

    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME (General linker)
    "ASSOCIATED_WITH",    # Fallback for weak correlations
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
]

def add_neo4j(final_results,merge_query,URI,AUTH):
  x=json.dumps(final_results)
  x=json.loads(x)
  with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")
    #record,summary,key=driver.execute_query("MATCH (p:Person) RETURN p.name",database_="neo4j")  ---------------->>>> for read operation
    record,summary,key=driver.execute_query("MATCH(n) DETACH DELETE n",database_="neo4j")
    for i in x:
      for j in i['relations']:
        source_name=j['source_entity'].get('name')
        source_type=j['source_entity'].get('type')
        target_name=j['target_entity'].get('name')
        target_type=j['target_entity'].get('type')
        relationship=j['relation_type']
        if not (source_name and target_name and relationship):
                continue
        if relationship in relationships:
          final_query=merge_query.replace("RELATIONSHIP",relationship)
          summary=driver.execute_query(final_query,name1=source_name,type1=source_type,name2=target_name,type2=target_type,database_="neo4j")
        else:
          print("----LLM HALLUCINATED AND NOT FOLLOWED ORDERS-----------------------------------------------------")


def communties_extracton(g,URI,AUTH):
  with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")
    records,summary,key=driver.execute_query("MATCH (n:Entity)-[r]->(m:Entity) RETURN n,r,m")
  for record in records:
    x=record.data()
    source=x.get("n")
    source_nxname=source["name"]
    source_nxtype=source["type"]
    rel=x.get("r")
    rel=rel[1]
    target=x.get("m")
    target_nxname=target["name"]
    target_nxtype=target["type"]
    g.add_node(source_nxname)
    g.add_node(target_nxname)
    g.add_edge(source_nxname,target_nxname)
    g.nodes[source_nxname]["type"]=source_nxtype
    g.nodes[target_nxname]["type"]=target_nxtype
    g.edges[source_nxname,target_nxname]["name"]=rel
  print("done")
  communities = algorithms.leiden(g)
  return communities


system_template = """You are a specialized Medical Science Communicator.
Your task is to turn raw graph data into a professional, distinct, and grammatically fluent clinical summary.

### 1. THE CONSTRAINT (CLOSED WORLD)
- You may ONLY use the entities and relationships provided in the input.
- Do NOT bring in outside medical facts (no hallucinations).
- **VERBATIM ADHERENCE:** You must respect the *strength* of the relationship provided.

### 2. THE "ASSOCIATED_WITH" TRAP (CRITICAL)
- If the relationship is `ASSOCIATED_WITH`, `HAS_CONTEXT`, or `RELATED_TO`:
  - **DO NOT** imply causation (e.g., do NOT write "leads to", "causes", "triggers").
  - **DO NOT** invent a mechanism (e.g., do NOT write "via the activation of...").
  - **MUST USE** neutral language: "is linked to", "co-occurs with", "is present in the context of".
  - *Example:* If Input is `(Drug A) -> [ASSOCIATED_WITH] -> (Symptom B)`, it means "Drug A is associated with Symptom B." and nothing else (Do NOT write "Drug A causes Symptom B").

### 3. THE FLUENCY RULES (CRITICAL FOR GOOD GRAMMAR)
- **No Robotic Lists:** Do not write "A causes B. B causes C."
- **Use Compound Sentences:** Combine related edges into fluid thoughts.
  - *Bad:* "Obesity causes Diabetes. Diabetes causes Renal Failure."
  - *Good:* "Obesity is a driver of Diabetes , which subsequently leads to Renal Failure ."
- **Vary Sentence Structure:** Use transition words (e.g., "Furthermore," "Consequently," "In contrast").


Here is the strict graph data you must convert into text.

### PART A: KNOWN ENTITIES (node_list)
{node}

### PART B: VERIFIED EDGES (relationship_list)
{relationship}

### YOUR TASK:
Perform this task in two distinct steps using XML tags:

STEP 1: <thinking>
1. **Cluster Analysis:** Group the edges into logical themes (e.g., "Pathology Progression", "Treatment Regimen", "Symptoms").
2. **Chain Building:** Identify long paths (A -> B -> C) that can be merged into a single fluid sentence.
3. **Orphan Check:** Identify any isolated nodes and plan how to weave them into the context naturally.
4.""INCLUSION:** Try to include every node into the summary
5. **Fact Check:** Confirm that every planned sentence is supported by a specific edge in Part B.

STEP 2: <summary>
Write the precise, grounded clinical summary based on your plan. Retain the exact node name DO NOT CHANGE IT. DO NOT CREATE INFORMATON OR USE PRETRAINED KNOWLEDGE
Do not mention the graph structure (e.g., "Node A connects to Node B")—write as a doctor speaking to a doctor.
</summary>
"""

strict_graph_prompt = PromptTemplate(
    template=system_template,
    input_variables=["node","relationship"]
)



def five_seasons(g,strict_graph_prompt,communities):
  community_summary=[]
  pattern = r"<summary>(.*?)</summary>"
  llm = ChatGroq(
      model="llama-3.3-70b-versatile",
      temperature=0,
      max_retries=2,
  )
  chain2= strict_graph_prompt | llm
  for i,member in enumerate(communities.communities):
    community_relations=[]
    community_nodes=[]
    subgraph=g.subgraph(member)
    community_node=member
    for j in subgraph.edges:
      rel=subgraph.edges[j]["name"]
      relation=j[0]+" "+rel+" "+j[1]
      if relation not in community_relations:
        community_relations.append(relation)
    community_response=chain2.invoke({"node":community_node,"relationship":community_relations})
    match = re.search(pattern, community_response.content, re.DOTALL)
    if match:
      community_summary.append(match.group(1).strip())
    else:
      community_summary.append(community_response.content)
    if i>4:
      break
  return community_summary


def get_injection_nodes(G, partition, top_n=5, inject_count=10):
    community_map = defaultdict(list)
    for node, comm_id in partition.items():
        community_map[comm_id].append(node)
    sorted_communities = sorted(community_map.items(), key=lambda x: len(x[1]), reverse=True)
    top_community_ids = {comm_id for comm_id, nodes in sorted_communities[:top_n]}
    tail_nodes = []
    for comm_id, nodes in sorted_communities[top_n:]:
        tail_nodes.extend(nodes)

    if not tail_nodes:
        return []
    pagerank_scores = nx.pagerank(G)
    tail_scores = {node: pagerank_scores[node] for node in tail_nodes}
    sorted_tail_nodes = sorted(tail_scores.items(), key=lambda x: x[1], reverse=True)
    injection_list = [node for node, score in sorted_tail_nodes[:inject_count]]
    return injection_list


INJECTION_PROMPT = PromptTemplate(template="""
You are a Medical Data Analyst.
You have been provided with a list of "orphan" medical entities extracted from the tail end of a patient's knowledge graph. These entities are disconnected from the main clinical narrative but are crucial for completeness.

### INPUT DATA (Raw Entity List)
{raw_node_list}

### YOUR TASK
Convert this raw list into a structured "Additional Clinical Findings" section.

### GUIDELINES
1. **Filter Noise:** Remove any non-medical terms (e.g., "Date", "Page 2", "Hospital Name", generic words like "Patient"). Keep ONLY distinct clinical concepts.
2. **Categorize:** Group the entities logically (e.g., "Medications," "Comorbidities," "Lab Values," "Symptoms").
3. **Neutral Phrasing:** Do NOT invent relationships.
   - *Bad:* "The patient took Aspirin because of the Headache." (Hallucination risk)
   - *Good:* "The dataset also notes the presence of Aspirin and Headache."
4. **Keyword Preservation:** Ensure the exact names of the important entities appear in the output to maintain semantic fidelity.

### OUTPUT FORMAT
Provide the output in a single block using the following structure:

**Additional Clinical Notations**
Analysis of the remaining graph data highlights several isolated clinical concepts.
* **[Category 1]:** [Entity A], [Entity B]
* **[Category 2]:** [Entity C], [Entity D]
* **Other Findings:** [Entity E]
""",input_variables=["raw_node_list"])




final_summary_template = PromptTemplate(
    template='''
You are a Medical Archivist and Clinical Compiler.
You have received several partial reports and a list of isolated entities.
Your goal is to weave EVERY single fact from these inputs into one master document.

### 1. THE "ZERO COMPRESSION" MANDATE
- **DO NOT SUMMARIZE.** Your output should be roughly the same length as the combined inputs.
- **DO NOT DELETE DETAILS.** If Input A mentions "left ventricular hypertrophy" and Input B mentions "LVEF 45%", the final report MUST contain both.
- **REDUNDANCY CHECK:** Only remove information if it is an *exact* word-for-word repetition. If it adds even a tiny nuance, KEEP IT.

### 2. THE "DIRECT REUSE" RULE
- You are strictly required to **COPY AND PASTE** complete sentences from the input if they are grammatically sound.
- Do not rewrite a perfect medical sentence just to sound different. Preserve the original phrasing to ensure accuracy.

### 3. INPUT DATA
"""
{community_summaries_text}
"""

### 4. THE COMPILATION TASK (Chain of Thought)
Perform this in two steps using XML tags(MAKE SURE YOU USE THEM):

STEP 1: <thinking>
1. **Scope Audit:** Scan the inputs. Acknowledge that you must preserve ALL specific metrics (dates, dosages, counts).
2. **Structure Design:** organizing the "Narrative" inputs into a logical flow (e.g., Clinical Presentation -> Diagnostics -> Therapeutic Interventions).
3. **Integration of Isolated Nodes:** Look at the "isolated/tail" list (the non-sentence part of the input). Can any of these items be naturally inserted into the narrative sections?
   - *Yes:* Plan to insert them where they fit.
   - *No:* Plan to list them in the "Additional Findings" section.
4. **Safety Check:** Confirm you are not adding any outside information.

STEP 2: <output>
Write the Master Clinical Report.(DO NOT ADD ANYTHING IRRELEVANT)
- **Section 1: Comprehensive Clinical Narrative** (A dense, detailed account. Use transition words to stitch the input sentences together, but do not shorten them).
- **Section 2: Additional Clinical Notations** (List any remaining entities from the isolated list that could not be woven into Section 1).
</output>

BEGIN:
    ''',
    input_variables=["community_summaries_text"]
)



In [ ]:
### pydantic fail logic
import json
import re

def extract_relations_from_error(e):
    failed_gen = None

    if hasattr(e, "error") and isinstance(e.error, dict):
        failed_gen = e.error.get("failed_generation")
    if failed_gen is None and hasattr(e, "body"):
        body = e.body
        if isinstance(body, dict):
            failed_gen = body.get("error", {}).get("failed_generation")
    if failed_gen is None and e.args:
        arg0 = e.args[0]
        if isinstance(arg0, dict):
            failed_gen = arg0.get("error", {}).get("failed_generation")

    if failed_gen is None:
    
        text = str(e)
        match = re.search(r"<function=MedicalGraph>(.*?)</function>", text, re.DOTALL)
        if match:
            failed_gen = match.group(1)

    if failed_gen is None:
        raise ValueError("failed_generation not found anywhere")

    start = failed_gen.find("{")
    end = failed_gen.rfind("}") + 1
    json_str = failed_gen[start:end]
    try:
        data = json.loads(json_str, strict=False)
    except json.JSONDecodeError:
        try:
            sanitized_str = json_str.replace('\n', ' ').replace('\r', '')
            data = json.loads(sanitized_str, strict=False)
        except Exception as final_e:
            print(f"CRITICAL JSON FAILURE. RAW STRING:\n{json_str}")
            raise final_e


    return [
        {
            "source_entity": r["source_entity"]["name"],
            "source_type": r["source_entity"].get("type", "OTHER"),
            "relation": r["relation_type"],
            "target_entity": r["target_entity"]["name"],
            "target_type": r["target_entity"].get("type", "OTHER"),
            "evidence": r.get("evidence", ""),
            "confidence": r.get("confidence")
        }
        for r in data.get("relations", [])
    ]
VALID_RELATIONS = {
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION
    "EXHIBITS",          # PATHOGEN -> VARIANT
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN
    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST
    "DIFFERS_FROM",      # VARIANT -> VARIANT
    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME
    "ASSOCIATED_WITH",   # Fallback (weak correlation)
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
}
def normalize_relation(relation: str) -> str:
    if relation in VALID_RELATIONS:
        return relation
    return "ASSOCIATED_WITH"
def normalize_relations(relations):
    normalized = []
    for r in relations:
        original = r["relation"]
        fixed = normalize_relation(original)
        if original != fixed:
            print(f"[RELATION FIX] {original} → {fixed}") ###delete this if want to
        r["relation"] = fixed
        normalized.append(r)
    return normalized
def convert_relations_to_canonical(relations):
    canonical_block = {"relations": []}
    for r in relations:
        canonical_block["relations"].append({
            "source_entity": {
                "name": r["source_entity"],
                "type": r.get("source_type", "OTHER")
            },
            "relation_type": r["relation"],
            "target_entity": {
                "name": r["target_entity"],
                "type": r.get("target_type", "OTHER")
            },
            "evidence": r.get("evidence", ""),
            "confidence": float(r.get("confidence", 0.7))
        })
    return canonical_block

In [ ]:
current_abstract=0 ### change this accordingly
from pydantic import ValidationError

folder_path = '/content/drive/MyDrive/Graph_Rag_Testing'
file_name='COT.xlsx'
if os.path.exists(folder_path):
  file_path=os.path.join(folder_path,file_name)
  df=pd.read_excel(file_path)

while current_abstract < len(df):
  chunks=chunked(df.iloc[current_abstract,0])
  llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)

  structured_llm = llm.with_structured_output(MedicalGraph)

  chain = prompt_template | structured_llm
  final_results = []
  errors = []

  print(f"Starting extraction on {len(chunks)} chunks...")

  for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
    while True:
        try:
            result = chain.invoke({
                "examples": few_shot_examples,
                "text": chunk
            })

            final_results.append(result.model_dump())
            break

        except RateLimitError:
          if current_key<len(groq_key)-1:
            current_key+=1
            print("KEY CHANGED-----------------------------------------------,current key:",current_key)
          else:
            runtime.unassign()
            time.sleep(10)
            sys.exit()
          os.environ["GROQ_API_KEY"]=groq_key[current_key]
          llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)
          structured_llm = llm.with_structured_output(MedicalGraph)
          chain = prompt_template | structured_llm

        except Exception as e:
            print(f"Error on chunk {i}: {e}")
            errors.append({"chunk_index": i, "error": str(e)})
            relations = extract_relations_from_error(e)
            relations = normalize_relations(relations)
            canonical = convert_relations_to_canonical(relations)
            final_results.append(canonical)
            print("Successful")
            break
        time.sleep(4)

  add_neo4j(final_results,merge_query,URI,AUTH)
  g=nx.Graph()
  communities=communties_extracton(g,URI,AUTH)
  while True:
    try:
      community_summary=five_seasons(g,strict_graph_prompt,communities)
      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED-----------------------------------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  while True:
    try:
      communities_injection = algorithms.leiden(g)
      partition = {}
      for i, community_list in enumerate(communities_injection.communities):
        for node in community_list:
          partition[node] = i
      injected_nodes = get_injection_nodes(g, partition, top_n=5, inject_count=15)
      llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)
      chain_inject= INJECTION_PROMPT | llm
      injection_response=chain_inject.invoke({"raw_node_list":injected_nodes})
      community_summary.append(injection_response.content)

      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED-----------------------------------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  while True:
    try:
      llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)
      chain3= final_summary_template | llm
      final_response=chain3.invoke({"community_summaries_text":community_summary})
      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED------------------------&&&&&&^^^^^^^%%%%%%%%%%%-----------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  pattern = r"<output>(.*?)</output>"
  match = re.search(pattern, final_response.content, re.DOTALL)
  if match:
    print(match.group(1).strip())
    df.at[current_abstract, 'summary']=match.group(1).strip()
  else:
    print(final_response.content)
    df.at[current_abstract, 'summary']=final_response.content
  df.to_excel(file_path,index=False)
  current_abstract+=1
  print("UPDATED SUCESSFULLY------------------------$$$$$$$$$$$$$$$%%%%%%%%%%%%%%%%%%%%-----------------------------")


In [7]:
current_abstract=19### change this accordingly


folder_path = '/content/drive/MyDrive/Graph_Rag_Testing'
file_name='COT.xlsx'
if os.path.exists(folder_path):
  file_path=os.path.join(folder_path,file_name)
  df=pd.read_excel(file_path)
  print(df.iloc[current_abstract,2])

nan


#UPGRADE

In [ ]:
!pip install langchain
!pip install langchain_text_splitters
!pip install openai
!pip install -U "langchain[google-genai]"
!pip install pydantic
!pip install langchain-groq
!pip install neo4j
!pip install networkx
!pip install pyvis
!pip install cdlib
!pip install leidenalg

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from groq import RateLimitError
import json
from google.colab import runtime
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chat_models import init_chat_model
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel,Field
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from neo4j import GraphDatabase
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network
from IPython.display import display, HTML
from cdlib import algorithms
from typing import List, Literal,Union
import time
from tqdm import tqdm # Progress bar
from langchain_core.output_parsers import PydanticOutputParser
import sys
from google.colab import drive
import pandas as pd
import re
from collections import defaultdict


In [4]:
def chunked(text):
  splitter = RecursiveCharacterTextSplitter(
      chunk_size=500,
      chunk_overlap=100,
      separators=["\n\n", "\n", " ", ""])
  chunks=splitter.split_text(text)
  return chunks


#### ENTITY RELATION EXTRACTION RULES
EntityType = Literal[
    # CLASS A: BIOLOGICAL AGENTS
    "PATHOGEN",          # Viruses, Bacteria, Fungi (e.g., H3N2, E. coli)
    "HOST",              # Patients, Animals, Demographics (e.g., Refugee patients, Children, Ferrets)

    # CLASS B: MOLECULAR REALITY (Crucial for your data)
    "GENETIC_ENTITY",    # Genes, RNA, DNA, Primers, Amplicons (e.g., HA Gene, PCR Primers)
    "MOLECULAR_VARIANT", # Mutations, Substitutions, Strains, Isolates (e.g., K145N, Nepal Isolates)
    "CHEMICAL",          # Drugs, Proteins, Reagents, Antibodies (e.g., Lisinopril, HA Protein)

    # CLASS C: CLINICAL REALITY
    "CONDITION",         # Diseases, Symptoms, Syndromes (e.g., Influenza, Fever)
    "PHENOMENON",        # Biological processes (e.g., Genetic Drift, Antigenic Variation)

    # CLASS D: INTERVENTION & CONTEXT
    "PROCEDURE",         # Tests, Assays, Sequencing, Vaccinations (e.g., PCR, HI Assay)
    "DEVICE",            # Machines, Kits (e.g., Magnapure LX)
    "LOCATION",          # Geographic or Anatomical (e.g., Nepal, Throat)
    "METRIC",            # Measurements, Titers (e.g., 1:320, 4-fold increase)
    "CELL",
    "ANATOMY",          # Added these 2
    "OTHER"              # Strict fallback
]

RelationType = Literal[
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION (H3N2 causes Influenza)
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION (Tamiflu treats Influenza)
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION (Vaccine prevents Flu)

    "EXHIBITS",          # PATHOGEN -> VARIANT (H3N2 exhibits K145N mutation)
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY (PCR detects RNA)
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN (Primers target HA Gene)

    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST (Nepal Isolates originate from Nepal)
    "DIFFERS_FROM",      # VARIANT -> VARIANT (Strain A differs from Strain B)

    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME (General linker)
    "ASSOCIATED_WITH",    # Fallback for weak correlations
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
]
StudySection = Literal[
    "INTRODUCTION",      # Background, Hypothesis
    "METHODS",           # Study Design, Procedures
    "RESULTS",           # Raw Findings, Data
    "DISCUSSION",        # Interpretation, Limitations
    "CONCLUSION",        # Final Takeaways, Future Work
    "UNKNOWN"            # Fallback
]

class Entity(BaseModel):
    name: str = Field(description="The precise scientific name. For 'Nepal Isolates', extract 'Isolates' as the name and 'Nepal' as a separate Location node.")
    type: EntityType = Field(description="The strict scientific category.If unsure of the category, STRICTLY use 'OTHER'.")

class Relation(BaseModel):
    source_entity: Entity = Field(description="The subject of the relation")
    relation_type: RelationType = Field(description="The strict scientific verb connecting them.If the relationship is complex, you MUST select 'ASSOCIATED_WITH'.")
    target_entity: Entity = Field(description="The object of the relation")
    modality: Literal["CONFIRMED", "HYPOTHETICAL", "NEGATED", "ASSOCIATED"] = Field(
        description="CONFIRMED: Fact. HYPOTHETICAL: 'Suggests/May'. NEGATED: 'Does NOT'. ASSOCIATED: 'Linked to'."
    )
    nuance_comment: str = Field(
        description=
            "A brief, 1-sentence explanation of the relationship's context. "
            "This is where you explain the 'How/Why'."
            "Anything you were not able to mention in relation , DO NOT CREATE INFORMATION"

    )
    study_section: StudySection = Field(
        description="Which part of the scientific paper did this fact come from?"
    )
    evidence: str = Field(description="The EXACT quote from the text supporting this.")
    confidence: float = Field(description="Confidence score (0.0-1.0).")

class MedicalGraph(BaseModel):
   reasoning_trace: str = Field(
        description='''
            "THINKING STEP-BY-STEP LOGIC:\n"
            "Carefully think before answering"
            "DO NOT CREATE INFORMATION , USE INFORMATION PROVIDED IN THE TEXT ONLY."
            ""CONNECTIVITY PLANNING:\n"
            "1. IDENTIFY THE HUB: Which entity is the main subject? (e.g., 'The Patient' or 'The Virus').\n"
            "2. LINK ORPHANS: Look for entities that might become isolated. explicitly plan how to link them back to the Hub.\n"
            '''

    )
   relations: List[Relation]




##### FIRST PROMPT

system_prompt = """You are an expert Medical Knowledge Graph Engineer.
Your goal is to extract a highly precise, scientifically accurate Knowledge Graph from the provided text.

### CORE INSTRUCTION: DECONSTRUCTION OVER LABELING
Medical text often compresses complex relationships into single noun phrases.
Do NOT label complex phrases as single nodes. Instead, DECONSTRUCT them into atomic components.

Examples of Deconstruction:
- "Metastatic Lung Cancer" -> "Lung Cancer" (CONDITION) + "Metastasis" (PHENOMENON).
- "Nepal Isolates" -> "Isolates" (MOLECULAR_VARIANT) + "Nepal" (LOCATION).
- "PCR Primers" -> "Primers" (GENETIC_ENTITY) + "PCR" (PROCEDURE).

### STRICT ENTITY GUIDELINES
1. PATHOGEN: Only the active infectious agent (e.g., "H3N2", "S. aureus"). Do not confuse with the Disease.
2. CONDITION: The passive disease or symptom (e.g., "Influenza", "Pneumonia").
3. CHEMICAL: Includes drugs, proteins, and lab reagents.
4. MOLECULAR_VARIANT: Use this for specific strains, mutations, or receptor statuses (e.g., "Delta Variant", "HER2+").
5. GENETIC_ENTITY: Use this for Genes, RNA, DNA, and Primers.
6. LOCATION: Use this for both Geography (Countries) and Anatomy (Body parts).

**"ASSOCIATED_WITH" TRAP:**
If the relationship is weak or statistical, do NOT use `CAUSES` or `TREATS`. Use `ASSOCIATED_WITH` or `CORRELATES_WITH` and set modality to `ASSOCIATED`.

### RELATIONSHIP USAGE RULES
- Use 'EXHIBITS' to link a Subject to its Properties or Variants (e.g., Virus -> EXHIBITS -> Mutation).
- Use 'TARGETS' for Mechanism of Action (e.g., Drug -> TARGETS -> Protein).
- Use 'ORIGINATES_FROM' to link samples/variants to their source (e.g., Tumor -> ORIGINATES_FROM -> Tissue).
- Use 'HAS_CONTEXT' to link Entities to Metrics, Time, or Locations (e.g., Patient -> HAS_CONTEXT -> Age 60).
- Use 'DETECTS' for Diagnostic tools finding a Condition or Pathogen.

### RELATIONSHIP & MODALITY RULES (THE HALLUCINATION KILLER)
You must select the correct `modality` for every relationship.
- **CONFIRMED:** The text explicitly states a fact. (e.g., "Drug X inhibits Protein Y").
- **HYPOTHETICAL:** The text uses hedging language. (e.g., "Suggests", "May", "Could", "Potential target").
- **NEGATED:** The text explicitly denies a link. (e.g., "No correlation found", "Did not treat").
- **ASSOCIATED:** The text implies a link without mechanism. (e.g., "X is linked to Y", "X co-occurs with Y").

### FIELD USAGE GUIDELINES
- **`nuance_comment`**: This is your "Context Dump".
  - Use it for: Specific conditions ("in mice only"), contradictions ("conflicts with Smith et al"), or statistical details ("p < 0.05").
  - Do NOT create new info, but preserve the richness here.
- **`study_section`**: Infer the section based on tone.
  - "We hypothesize..." -> INTRODUCTION.
  - "We measured..." -> METHODS.
  - "Data showed..." -> RESULTS.
  - "This suggests..." -> DISCUSSION/CONCLUSION.

### CONNECTIVITY PLANNING (`reasoning_trace`)
Before generating relations, use the `reasoning_trace` field to:
1.  **Identify the Hub:** What is the central topic?
2.  **Plan Connectivity:** Avoid orphan nodes.Ensure every extracted node links back to the hub or a main chain.

### OUTPUT LOGIC
- If the text mentions "Resistance to Tamiflu", extract:
  (Tamiflu: CHEMICAL) -[TREATS]-> (Influenza: CONDITION)
  (Influenza: CONDITION) -[EXHIBITS]-> (Resistance: PHENOMENON)
  (Resistance: PHENOMENON) -[ASSOCIATED_WITH]-> (Tamiflu: CHEMICAL)

Analyze the text deeply. Bias towards granular, scientifically precise nodes.
"""

few_shot_examples = """
Input: "Our study suggests that H3N2 mutations in Nepal isolates might cause resistance to Oseltamivir."
Output: {
  "reasoning_trace": "Hub: H3N2. Deconstruction: 'Nepal isolates' -> 'Isolates' + 'Nepal'. Modality Check: 'Suggests/Might' -> HYPOTHETICAL.",
  "relations": [
    {
      "source_entity": {"name": "Isolates", "type": "MOLECULAR_VARIANT"},
      "relation_type": "ORIGINATES_FROM",
      "target_entity": {"name": "Nepal", "type": "LOCATION"},
      "modality": "CONFIRMED",
      "nuance_comment": "Specific strain origin",
      "study_section": "RESULTS",
      "evidence": "Nepal isolates",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": "H3N2", "type": "PATHOGEN"},
      "relation_type": "EXHIBITS",
      "target_entity": {"name": "Mutation", "type": "MOLECULAR_VARIANT"},
      "modality": "CONFIRMED",
      "nuance_comment": "",
      "study_section": "RESULTS",
      "evidence": "H3N2 mutations",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": "Mutation", "type": "MOLECULAR_VARIANT"},
      "relation_type": "CAUSES",
      "target_entity": {"name": "Resistance", "type": "PHENOMENON"},
      "modality": "HYPOTHETICAL",
      "nuance_comment": "Text says 'might cause', indicating uncertainty.",
      "study_section": "CONCLUSION",
      "evidence": "mutations... might cause resistance",
      "confidence": 0.7
    },
    {
      "source_entity": {"name": "Resistance", "type": "PHENOMENON"},
      "relation_type": "ASSOCIATED_WITH",
      "target_entity": {"name": "Oseltamivir", "type": "CHEMICAL"},
      "modality": "CONFIRMED",
      "nuance_comment": "Context of resistance",
      "study_section": "CONCLUSION",
      "evidence": "resistance to Oseltamivir",
      "confidence": 0.9
    }
  ]
}

Input: "Patient denies chest pain but reports severe shortness of breath."
Output: {
  "reasoning_trace": "Hub: Patient. Negation Check: 'denies chest pain' -> Modality NEGATED. Attribute: 'severe' -> nuance_comment.",
  "relations": [
    {
      "source_entity": {"name": "Patient", "type": "HOST"},
      "relation_type": "EXHIBITS",
      "target_entity": {"name": "Chest Pain", "type": "SYMPTOM"},
      "modality": "NEGATED",
      "nuance_comment": "Patient explicitly denied this symptom.",
      "study_section": "RESULTS",
      "evidence": "denies chest pain",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": "Patient", "type": "HOST"},
      "relation_type": "EXHIBITS",
      "target_entity": {"name": "Shortness of Breath", "type": "SYMPTOM"},
      "modality": "CONFIRMED",
      "nuance_comment": "Described as severe.",
      "study_section": "RESULTS",
      "evidence": "reports severe shortness of breath",
      "confidence": 1.0
    }
  ]
}
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Here are examples of the expected JSON output format:\n{examples}"),
    ("human", "Analyze this clinical text and extract the graph:\n{text}")
])


####ADDING NEO4J

merge_query="""
MERGE (e:Entity{name:$name1})
ON CREATE
  SET e.type=$type1

MERGE (f:Entity{name:$name2})
ON CREATE
  SET f.type=$type2

MERGE (e)-[r:RELATIONSHIP]->(f)
ON CREATE
  SET r.modality=$modality,
      r.Nuance=$nuance,
      r.study_section=$study,
      r.evidence=$evidence
"""
relationships=[
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION (H3N2 causes Influenza)
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION (Tamiflu treats Influenza)
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION (Vaccine prevents Flu)

    "EXHIBITS",          # PATHOGEN -> VARIANT (H3N2 exhibits K145N mutation)
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY (PCR detects RNA)
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN (Primers target HA Gene)

    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST (Nepal Isolates originate from Nepal)
    "DIFFERS_FROM",      # VARIANT -> VARIANT (Strain A differs from Strain B)

    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME (General linker)
    "ASSOCIATED_WITH",    # Fallback for weak correlations
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
]

def add_neo4j(final_results,merge_query,URI,AUTH):
  x=json.dumps(final_results)
  x=json.loads(x)
  with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")
    #record,summary,key=driver.execute_query("MATCH (p:Person) RETURN p.name",database_="neo4j")  ---------------->>>> for read operation
    record,summary,key=driver.execute_query("MATCH(n) DETACH DELETE n",database_="neo4j")
    for i in x:
      for j in i['relations']:
        source_name=j['source_entity'].get('name')
        source_type=j['source_entity'].get('type')
        target_name=j['target_entity'].get('name')
        target_type=j['target_entity'].get('type')
        relationship=j['relation_type']
        modality=j['modality']
        nuance=j['nuance_comment']
        study_section=j['study_section']
        evidence=j['evidence']
        if not (source_name and target_name and relationship):
                continue
        if relationship in relationships:
          final_query=merge_query.replace("RELATIONSHIP",relationship)
          summary=driver.execute_query(final_query,name1=source_name,type1=source_type,name2=target_name,type2=target_type,modality=modality,nuance=nuance,study=study_section,evidence=evidence,database_="neo4j")
        else:
          print("----LLM HALLUCINATED AND NOT FOLLOWED ORDERS-----------------------------------------------------")


### EXTRACTING BACK FROM NEO4J
def communties_extracton(g,URI,AUTH):
  with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")
    records,summary,key=driver.execute_query("MATCH (n:Entity)-[r]->(m:Entity) RETURN n,r,m,r.modality,r.Nuance,r.study_section")
  for record in records:
    x=record.data()
    source=x.get("n")
    source_nxname=str(source["name"])
    source_nxtype=source["type"]
    rel=x.get("r")
    rel=rel[1]
    target=x.get("m")
    target_nxname=str(target["name"])
    target_nxtype=target["type"]
    rel_modal=x.get('r.modality')
    rel_nuance=x.get('r.Nuance')
    rel_study=x.get('r.study_section')
    g.add_node(source_nxname)
    g.add_node(target_nxname)
    g.add_edge(source_nxname,target_nxname)
    g.nodes[source_nxname]["type"]=source_nxtype
    g.nodes[target_nxname]["type"]=target_nxtype
    g.edges[source_nxname,target_nxname]["name"]=rel
    g.edges[source_nxname,target_nxname]["Modality"]=rel_modal
    g.edges[source_nxname,target_nxname]["Nuance"]=rel_nuance
    g.edges[source_nxname,target_nxname]["Study_Section"]=rel_study
  print("done")
  communities = algorithms.leiden(g)
  return communities


system_template = """You are a specialized Medical Science Communicator.
Your task is to turn raw graph data into a professional, distinct, and grammatically fluent clinical summary.

### 1. THE CONSTRAINT (CLOSED WORLD)
- You may ONLY use the entities and relationships provided in the input.
- Do NOT bring in outside medical facts (no hallucinations). DO NOT USE PRETRAINED MEDICAL KNOWLEDGE
- **VERBATIM ADHERENCE:** You must respect the *strength* of the relationship provided.

### 2. THE "ASSOCIATED_WITH" TRAP (CRITICAL)
- If the relationship is `ASSOCIATED_WITH`, `HAS_CONTEXT`, or `RELATED_TO`:
  - **DO NOT** imply causation (e.g., do NOT write "leads to", "causes", "triggers").
  - **DO NOT** invent a mechanism (e.g., do NOT write "via the activation of...").
  - **MUST USE** neutral language: "is linked to", "co-occurs with", "is present in the context of".
  - *Example:* If Input is `(Drug A) -> [ASSOCIATED_WITH] -> (Symptom B)`, it means "Drug A is associated with Symptom B." and nothing else (Do NOT write "Drug A causes Symptom B").

### 3. THE FLUENCY RULES (CRITICAL FOR GOOD GRAMMAR)
- **No Robotic Lists:** Do not write "A causes B. B causes C."
- **Use Compound Sentences:** Combine related edges into fluid thoughts.
  - *Bad:* "Obesity causes Diabetes. Diabetes causes Renal Failure."
  - *Good:* "Obesity is a driver of Diabetes , which subsequently leads to Renal Failure ."
- **Vary Sentence Structure:** Use transition words (e.g., "Furthermore," "Consequently," "In contrast").

### HOW TO DECODE THE INPUT (CRITICAL)
The input format is: `(Node A) -[RELATION]-> (Node B) <MODALITY: M | STUDY_SECTION: S | NUANCE: N>`

**A. MODALITY RULES (The Truth Filter):**
- **CONFIRMED:** State as fact. ("A causes B").
- **HYPOTHETICAL:** Use hedging. ("A *may* cause B", "Evidence suggests A links to B").
- **NEGATED:** Explicitly state the lack of link. ("A does *not* cause B", "A is ruled out").
- **ASSOCIATED:** Use neutral language. ("A is observed alongside B").

**B. THE "NUANCE" FIELD (The Context):**
- You MUST incorporate the `Note` content into the sentence to add depth.
- *Example:* Input: `(Drug) -> [TREATS] -> (Disease) <Note: in mice only>`
- *Output:* "The drug treats the disease *in murine models*."

**C. THE "STUDY_SECTION" FIELD (The Narrative Frame):**
- Use the `Section` to frame *when* the information is presented.
- *INTRODUCTION:* "Background data indicates..."
- *RESULTS:* "Clinical findings show..."
- *CONCLUSION:* "The study concludes that..."


Here is the strict graph data you must convert into text.

### PART A: KNOWN ENTITIES (node_list)
{node}

### PART B: VERIFIED EDGES (relationship_list)
{relationship}

### YOUR TASK:
Perform this task in two distinct steps using XML tags:

STEP 1: <thinking>
1. **Cluster Analysis:** Group the edges into logical themes (e.g., "Pathology Progression", "Treatment Regimen", "Symptoms").
2. **Chain Building:** Identify long paths (A -> B -> C) that can be merged into a single fluid sentence.
3. **Orphan Check:** Identify any isolated nodes and plan how to weave them into the context naturally.
4.""INCLUSION:** Try to include every node into the summary
5. **Fact Check:** Confirm that every planned sentence is supported by a specific edge in Part B.

STEP 2: <summary>
Write the precise, grounded clinical summary based on your plan. Retain the exact node name DO NOT CHANGE IT. DO NOT CREATE INFORMATON OR USE PRETRAINED KNOWLEDGE. Try to use all the NUMERIC metrics provided as it is DO NOT CHANGE THE METRICS.
Do not mention the graph structure (e.g., "Node A connects to Node B")—write as a doctor speaking to a doctor.
</summary>
"""

strict_graph_prompt = PromptTemplate(
    template=system_template,
    input_variables=["node","relationship"]
)



def five_seasons(g,strict_graph_prompt,communities):
  community_summary=[]
  pattern = r"<summary>(.*?)</summary>"
  llm = ChatGroq(
      model="llama-3.3-70b-versatile",
      temperature=0,
      max_retries=2,
  )
  chain2= strict_graph_prompt | llm
  for i,member in enumerate(communities.communities):
    community_relations=[]
    community_nodes=[]
    subgraph=g.subgraph(member)
    community_node=member
    for j in subgraph.edges:
      rel=subgraph.edges[j]["name"]
      rel_modal=subgraph.edges[j]["Modality"]
      rel_nuance=subgraph.edges[j]["Nuance"]
      rel_study=subgraph.edges[j]["Study_Section"]
      relation=j[0]+" "+rel+" "+j[1]+"METADATA FOR RELATION< MODALITY: "+rel_modal+" | STUDY_SECTION: "+ rel_study+" | NUANCE: "+rel_nuance+">"
      if relation not in community_relations:
        community_relations.append(relation)
    community_response=chain2.invoke({"node":community_node,"relationship":community_relations})
    match = re.search(pattern, community_response.content, re.DOTALL)
    if match:
      community_summary.append(match.group(1).strip())
    else:
      community_summary.append(community_response.content)
    if i>4:
      break
  return community_summary






def get_injection_nodes(G, partition, top_n=5, inject_count=10):
    community_map = defaultdict(list)
    for node, comm_id in partition.items():
        community_map[comm_id].append(node)
    sorted_communities = sorted(community_map.items(), key=lambda x: len(x[1]), reverse=True)
    top_community_ids = {comm_id for comm_id, nodes in sorted_communities[:top_n]}
    tail_nodes = []
    for comm_id, nodes in sorted_communities[top_n:]:
        tail_nodes.extend(nodes)

    if not tail_nodes:
        return []
    pagerank_scores = nx.pagerank(G)
    tail_scores = {node: pagerank_scores[node] for node in tail_nodes}
    sorted_tail_nodes = sorted(tail_scores.items(), key=lambda x: x[1], reverse=True)
    injection_list = [node for node, score in sorted_tail_nodes[:inject_count]]
    return injection_list


INJECTION_PROMPT = PromptTemplate(template="""
You are a Medical Data Analyst.
You have been provided with a list of "orphan" medical entities extracted from the tail end of a patient's knowledge graph. These entities are disconnected from the main clinical narrative but are crucial for completeness.

### INPUT DATA (Raw Entity List)
{raw_node_list}

### YOUR TASK
Convert this raw list into a structured "Additional Clinical Findings" section.

### GUIDELINES
1. **Filter Noise:** Remove any non-medical terms (e.g., "Date", "Page 2", "Hospital Name", generic words like "Patient"). Keep ONLY distinct clinical concepts.
2. **Categorize:** Group the entities logically (e.g., "Medications," "Comorbidities," "Lab Values," "Symptoms").
3. **Neutral Phrasing:** Do NOT invent relationships.
   - *Bad:* "The patient took Aspirin because of the Headache." (Hallucination risk)
   - *Good:* "The dataset also notes the presence of Aspirin and Headache."
4. **Keyword Preservation:** Ensure the exact names of the important entities appear in the output to maintain semantic fidelity.

### OUTPUT FORMAT
Provide the output in a single block using the following structure:

**Additional Clinical Notations**
Analysis of the remaining graph data highlights several isolated clinical concepts.
* **[Category 1]:** [Entity A], [Entity B]
* **[Category 2]:** [Entity C], [Entity D]
* **Other Findings:** [Entity E]
""",input_variables=["raw_node_list"])




final_summary_template = PromptTemplate(
    template='''
You are a Medical Archivist and Clinical Compiler.
You have received several partial reports and a list of isolated entities.
Your goal is to weave EVERY single fact from these inputs into one master document.

### 1. THE "ZERO COMPRESSION" MANDATE
- **DO NOT SUMMARIZE.** Your output should be roughly the same length as the combined inputs.
- **DO NOT DELETE DETAILS.** If Input A mentions "left ventricular hypertrophy" and Input B mentions "LVEF 45%", the final report MUST contain both.
- **REDUNDANCY CHECK:** Only remove information if it is an *exact* word-for-word repetition. If it adds even a tiny nuance, KEEP IT.

### 2. THE "DIRECT REUSE" RULE
- You are strictly required to **COPY AND PASTE** complete sentences from the input if they are grammatically sound.
- Do not rewrite a perfect medical sentence just to sound different. Preserve the original phrasing to ensure accuracy.

### 3. INPUT DATA
"""
{community_summaries_text}
"""
### 4. REDUNDANCY CHECK
IF multiple inputs state the exact same fact (e.g., "Dose was 10mg"), do NOT simply delete the duplicates.INSTEAD, merge them into a single confirmed statement
IF Input A provides a general claim (e.g., "BP improved") and Input B provides a specific metric (e.g., "BP dropped by 10mmHg"), you MUST preserve the specific metric.
Never delete a sentence if it contains a number, p-value, sample size, or dosage that appears nowhere else in the text.
Even if the surrounding text seems redundant, the number itself is a unique data point that must be preserved.
NEVER EVER CREATE YOUR OWN INFORMATION , ALWAYS USE THE INPUT DATA

### 5. THE COMPILATION TASK (Chain of Thought)
Perform this in two steps using XML tags(MAKE SURE YOU USE THEM):

STEP 1: <thinking>
1. **Scope Audit:** Scan the inputs. Acknowledge that you must preserve ALL specific metrics (dates, dosages, counts).
2. **Structure Design:** organizing the "Narrative" inputs into a logical flow (e.g., Clinical Presentation -> Diagnostics -> Therapeutic Interventions).
3. **Integration of Isolated Nodes:** Look at the "isolated/tail" list (the non-sentence part of the input). Can any of these items be naturally inserted into the narrative sections?
   - *Yes:* Plan to insert them where they fit.
   - *No:* Plan to list them in the "Additional Findings" section.
4. **Safety Check:** Confirm you are not adding any outside information.

STEP 2: <output>
Write the Master Clinical Report.(DO NOT ADD ANYTHING IRRELEVANT)(**RETAIN ALL THE NUMERIC METRICS**)
- **Section 1: Comprehensive Clinical Narrative** (A dense, detailed account. Use transition words to stitch the input sentences together, but do not shorten them).
- **Section 2: Additional Clinical Notations** (List any remaining entities from the isolated list that could not be woven into Section 1).
</output>

BEGIN:
    ''',
    input_variables=["community_summaries_text"]
)


In [5]:
### pydantic fail logic
import json
import re

def extract_relations_from_error(e):
    failed_gen = None

    # 1. Extraction Strategy (Same as your code)
    if hasattr(e, "error") and isinstance(e.error, dict):
        failed_gen = e.error.get("failed_generation")
    if failed_gen is None and hasattr(e, "body"):
        body = e.body
        if isinstance(body, dict):
            failed_gen = body.get("error", {}).get("failed_generation")
    if failed_gen is None and e.args:
        arg0 = e.args[0]
        if isinstance(arg0, dict):
            failed_gen = arg0.get("error", {}).get("failed_generation")

    if failed_gen is None:
        # Fallback regex search
        text = str(e)
        match = re.search(r"<function=MedicalGraph>(.*?)</function>", text, re.DOTALL)
        if match:
            failed_gen = match.group(1)

    if failed_gen is None:
        raise ValueError("failed_generation not found anywhere")

    start = failed_gen.find("{")
    end = failed_gen.rfind("}") + 1
    json_str = failed_gen[start:end]
    try:
        data = json.loads(json_str, strict=False)
    except json.JSONDecodeError:
        try:
            sanitized_str = json_str.replace('\n', ' ').replace('\r', '')
            data = json.loads(sanitized_str, strict=False)
        except Exception as final_e:
            print(f"CRITICAL JSON FAILURE. RAW STRING:\n{json_str}")
            raise final_e


    return [
        {
            "source_entity": r["source_entity"]["name"],
            "source_type": r["source_entity"].get("type", "OTHER"),
            "relation": r["relation_type"],
            "target_entity": r["target_entity"]["name"],
            "target_type": r["target_entity"].get("type", "OTHER"),
            "evidence": r.get("evidence", ""),
            "confidence": r.get("confidence"),
            "modality":r.get("modality"),
            "nuance_comment":r.get("nuance_comment"),
            "study_section":r.get("study_section")

        }
        for r in data.get("relations", [])
    ]
VALID_RELATIONS = {
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITON
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION
    "EXHIBITS",          # PATHOGEN -> VARIANT
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENTIC_ENTITY
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN
    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST
    "DIFFERS_FROM",      # VARIANT -> VARIANT
    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME
    "ASSOCIATED_WITH",   # Fallback (weak correlation)
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
}
def normalize_relation(relation: str) -> str:
    if relation in VALID_RELATIONS:
        return relation
    return "ASSOCIATED_WITH"
def normalize_relations(relations):
    normalized = []
    for r in relations:
        original = r["relation"]
        fixed = normalize_relation(original)
        if original != fixed:
            print(f"[RELATION FIX] {original} → {fixed}") ###delete this if want to
        r["relation"] = fixed
        normalized.append(r)
    return normalized
def convert_relations_to_canonical(relations):
    canonical_block = {"relations": []}
    for r in relations:
        canonical_block["relations"].append({
            "source_entity": {
                "name": r["source_entity"],
                "type": r.get("source_type", "OTHER")
            },
            "relation_type": r["relation"],
            "target_entity": {
                "name": r["target_entity"],
                "type": r.get("target_type", "OTHER")
            },
            "evidence": r.get("evidence", ""),
            "confidence": float(r.get("confidence", 0.7)),
            "modality":r.get("modality"),
            "nuance_comment":r.get("nuance_comment"),
            "study_section":r.get("study_section")
        })
    return canonical_block

In [6]:
class EntityNumber(BaseModel):
    name: Union[int, float,str] = Field(
        description="The EXACT numerical value found in the text (e.g., 10, 0.05, 320). Do not round or summarize.If this is a 'Target Anchor', use the scientific term as a string."
    )
    type: EntityType = Field(
        description="The strict scientific category this number represents. If it is a measurement, use 'METRIC'."
    )

class RelationNumber(BaseModel):
    source_entity: EntityNumber = Field(
        description="The subject of the relation (usually the entity being measured or the baseline value)."
    )
    relation_type: RelationType = Field(
        description="The verb connecting the number to the target. Use 'MEASURES' or 'HAS_CONTEXT' for simple data points."
    )
    target_entity: EntityNumber = Field(
        description="The semantic anchor (e.g., 'Efficacy Rate') or the main topic node."
    )
    modality: Literal["CONFIRMED", "HYPOTHETICAL", "NEGATED", "ASSOCIATED"] = Field(
        description="CONFIRMED: Fact. HYPOTHETICAL: 'Suggests/May'. NEGATED: 'Does NOT'. ASSOCIATED: 'Linked to'."
    )
    nuance_comment: str = Field(
        description="MANDATORY: Link the number to its physical context (e.g., '120 is the systolic BP at rest'). Do not create info."
    )
    study_section: StudySection = Field(
        description="Which part of the scientific paper did this fact come from?"
    )
    evidence: str = Field(
        description="The EXACT quote containing the number and its immediate context."
    )
    confidence: float = Field(description="Confidence score (0.0-1.0).")

class MedicalGraphNumber(BaseModel):
    reasoning_trace: str = Field(
        description='''
            "THINKING STEP-BY-STEP LOGIC:\n"
            "1. SCAN: List every digit in the text.\n"
            "2. FILTER: Ignore citation years (e.g., 2023, 2024), Table numbers, and Page numbers.\n"
            "3. ANCHORING: For every number, identify the 'Anchor' (what it measures).\n"
            "4. NULL CHECK: If no clinical numbers found, return relations: []."
            '''
    )
    relations: List[RelationNumber]= Field(description="List of relations. Leave EMPTY if no numerical data is found.")


system_prompt_numerical = """You are an expert Medical Knowledge Graph Engineer. Your mission is to extract numerical values with forensic precision, ensuring they are anchored to clinical subjects to prevent graph fragmentation.

### THE CITATION & METADATA FILTER
- **STRICT EXCLUSION:** You MUST ignore all citation years (e.g., "Smith (2023)", "Zhao et al., 2024").
- **STRICT EXCLUSION:** Ignore Table, Figure, and Page references (e.g., "Table 1", "Fig. 2").
- **FOCUS:** Only extract numbers representing clinical data: p-values, dosages, patient counts, concentrations, ratios, and titers.

### THE ANCHORING RULE (Anti-Fragmentation)
To prevent "floating" numbers that the GraphRAG system cannot find, you must create a "Hub-and-Spoke" chain:
1. Every Number (int/float) must be a Source Entity.
2. Every Number MUST be linked to a "Target Anchor" (the scientific name of the metric, e.g., "Survival Rate").
3. That "Target Anchor" MUST then be linked to the "Main Subject" of the chunk (e.g., "H3N2" or "Metformin").

### CORE RULE: ZERO SMOOTHING
- Never replace a digit with a word (e.g., do not change "98%" to "almost all").
- If the text says "0.045", you must extract 0.045.
- If NO clinical numbers exist, return: {"reasoning_trace": "NO_NUMERICAL_DATA_FOUND", "relations": []}.

### RELATIONSHIP & MODALITY
- Use 'MEASURES' to link a Number to its Metric Anchor.
- Use 'HAS_CONTEXT' or 'ASSOCIATED_WITH' to link the Anchor to the Pathogen/Condition.
- **CONFIRMED:** Explicit facts.
- **HYPOTHETICAL:** "Suggests", "May", "Expected".

### CONNECTIVITY PLANNING (reasoning_trace)
1. **FILTER:** List all numbers found, then explicitly state which are citations (to be ignored) and which are clinical data.
2. **ANCHORING:** Map each clinical number to a specific metric name.
3. **HUB:** Identify the central clinical subject to link everything together.
"""

few_shot_examples_numerical = """
Input: "According to Zhao et al. (2024), the H3N2 viral load dropped significantly by 4.2 logs (p < 0.005) within 72 hours."
Output: {
  "reasoning_trace": "1. SCAN: 2024, 4.2, 0.005, 72. 2. FILTER: '2024' is a citation year (IGNORE). 3. HUB: H3N2. 4. ANCHORING: Link 4.2 to 'Viral Load Reduction', 0.005 to 'Statistical Significance', and 72 to 'Time Duration'.",
  "relations": [
    {
      "source_entity": {"name": 4.2, "type": "METRIC"},
      "relation_type": "MEASURES",
      "target_entity": {"name": "Viral Load Reduction", "type": "PHENOMENON"},
      "modality": "CONFIRMED",
      "nuance_comment": "Log base 10 reduction in H3N2 particles.",
      "study_section": "RESULTS",
      "evidence": "viral load dropped significantly by 4.2 logs",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": "Viral Load Reduction", "type": "PHENOMENON"},
      "relation_type": "HAS_CONTEXT",
      "target_entity": {"name": "H3N2", "type": "PATHOGEN"},
      "modality": "CONFIRMED",
      "nuance_comment": "Metric specifically measures H3N2 decline.",
      "study_section": "RESULTS",
      "evidence": "H3N2 viral load dropped",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": 0.005, "type": "METRIC"},
      "relation_type": "MEASURES",
      "target_entity": {"name": "P-Value", "type": "METRIC"},
      "modality": "CONFIRMED",
      "nuance_comment": "Statistical significance of the viral load drop.",
      "study_section": "RESULTS",
      "evidence": "p < 0.005",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": 72, "type": "METRIC"},
      "relation_type": "MEASURES",
      "target_entity": {"name": "Treatment Duration", "type": "METRIC"},
      "modality": "CONFIRMED",
      "nuance_comment": "Hours elapsed before measurement.",
      "study_section": "METHODS",
      "evidence": "within 72 hours",
      "confidence": 1.0
    }
  ]
}

Input: "We observed a 15% increase in resistance markers in the 2023 Nepal isolates compared to baseline."
Output: {
  "reasoning_trace": "1. SCAN: 15, 2023. 2. FILTER: '2023' is a temporal isolate identifier/year (IGNORE to prevent citation hubs). 3. HUB: Nepal Isolates. 4. ANCHORING: Link 15 to 'Resistance Increase'.",
  "relations": [
    {
      "source_entity": {"name": 15, "type": "METRIC"},
      "relation_type": "MEASURES",
      "target_entity": {"name": "Resistance Marker Increase", "type": "PHENOMENON"},
      "modality": "CONFIRMED",
      "nuance_comment": "Percentage increase in markers compared to baseline.",
      "study_section": "RESULTS",
      "evidence": "15% increase in resistance markers",
      "confidence": 1.0
    },
    {
      "source_entity": {"name": "Resistance Marker Increase", "type": "PHENOMENON"},
      "relation_type": "ASSOCIATED_WITH",
      "target_entity": {"name": "Isolates", "type": "MOLECULAR_VARIANT"},
      "modality": "CONFIRMED",
      "nuance_comment": "Relates to Nepal variant isolates.",
      "study_section": "RESULTS",
      "evidence": "resistance markers in the... Nepal isolates",
      "confidence": 0.95
    }
  ]
}

Input: "Previous literature reviews discuss the general mechanism of viral entry without providing new titration data."
Output: {
  "reasoning_trace": "NO_NUMERICAL_DATA_FOUND: No clinical digits or quantitative metrics found. Purely descriptive text.",
  "relations": []
}
"""

prompt_template_numerical = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Here are some examples of correct extraction: {examples}"),
    ("human", "Analyze this clinical note: {text}")
])

In [7]:
import re

def has_relevant_numbers(text: str) -> bool:
    """
    Returns True if the text contains:
    - Decimals (0.05)
    - Percentages (50%)
    - Math/Ranges (10-20, 1/2, 5*, +3)
    - Integers that are likely NOT years (matches 5, 100; ignores 2024, 1999)
    """
    pattern = r"""
        \d+\.\d+                # Matches Decimals (0.05, 12.5)
        |                       # OR
        %                       # Matches Percent signs
        |                       # OR
        \d+\s*[-+*/]\s*\d+      # Matches Ranges/Fractions/Math (10-20, 10/2, 5*5, 1+1)
        |                       # OR
        [-+]\d+                 # Matches Signed Numbers (-5, +10)
        |                       # OR
        \d+[-+*/]               # Matches Trailing Signs (10+, 5*, HER2+)
        |                       # OR
        :\d+                    # Matches Ratios (1:640)
        |                       # OR
        \b(?!(?:19|20)\d{2}\b)\d+\b  # Matches Integers, EXCLUDING Years (19xx, 20xx)
    """
    return bool(re.search(pattern, text, re.VERBOSE))

In [ ]:
current_abstract=0 ### change this accordingly
from pydantic import ValidationError

folder_path = '/content/drive/MyDrive/Graph_Rag_Testing'
file_name='COT.xlsx'
if os.path.exists(folder_path):
  file_path=os.path.join(folder_path,file_name)
  df=pd.read_excel(file_path)

while current_abstract < len(df):
  chunks=chunked(df.iloc[current_abstract,0])
  llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)

  structured_llm = llm.with_structured_output(MedicalGraph)

  chain = prompt_template | structured_llm

  structured_llm_numerical=llm.with_structured_output(MedicalGraphNumber)
  chain_numerical=prompt_template_numerical | structured_llm_numerical

  final_results = []
  errors = []

  print(f"Starting extraction on {len(chunks)} chunks...")

  for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
    while True:
        try:
            result = chain.invoke({
                "examples": few_shot_examples,
                "text": chunk
            })

            final_results.append(result.model_dump())
            break

        except RateLimitError:
          if current_key<len(groq_key)-1:
            current_key+=1
            print("KEY CHANGED-----------------------------------------------,current key:",current_key)
          else:
            runtime.unassign()
            time.sleep(10)
            sys.exit()
          os.environ["GROQ_API_KEY"]=groq_key[current_key]
          llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)
          structured_llm = llm.with_structured_output(MedicalGraph)
          chain = prompt_template | structured_llm

        except Exception as e:
          print(f"Error on chunk {i}: {e}")
          errors.append({"chunk_index": i, "error": str(e)})
          try:
            relations = extract_relations_from_error(e)
            relations = normalize_relations(relations)
            canonical = convert_relations_to_canonical(relations)
            final_results.append(canonical)
            print("Successful")
            break
          except Exception as e3:
            print("UnSuccessful")
            break
    while True:
        try:
          if not has_relevant_numbers(chunk):
            print("SAVED API COST")
            break
          print("EXTRACTING NUMBER IF ANY")
          result1 = chain_numerical.invoke({"examples": few_shot_examples_numerical,"text": chunk})
          final_results.append(result1.model_dump())
          break
        except RateLimitError:
          if current_key<len(groq_key)-1:
            current_key+=1
            print("KEY CHANGED-----------------------------------------------,current key:",current_key)
          else:
            runtime.unassign()
            time.sleep(10)
            sys.exit()
          os.environ["GROQ_API_KEY"]=groq_key[current_key]
          llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)
          structured_llm_numerical=llm.with_structured_output(MedicalGraphNumber)
          chain_numerical=prompt_template_numerical | structured_llm_numerical

        except Exception as e:
          print(f"Error on chunk {i}: {e}")
          errors.append({"chunk_index": i, "error": str(e)})
          break
    time.sleep(4)
  add_neo4j(final_results,merge_query,URI,AUTH)
  g=nx.Graph()
  communities=communties_extracton(g,URI,AUTH)
  while True:
    try:
      community_summary=five_seasons(g,strict_graph_prompt,communities)
      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED-----------------------------------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  while True:
    try:
      communities_injection = algorithms.leiden(g)
      partition = {}
      for i, community_list in enumerate(communities_injection.communities):
        for node in community_list:
          partition[node] = i
      injected_nodes = get_injection_nodes(g, partition, top_n=5, inject_count=15)
      llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)
      chain_inject= INJECTION_PROMPT | llm
      injection_response=chain_inject.invoke({"raw_node_list":injected_nodes})
      community_summary.append(injection_response.content)

      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED-----------------------------------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  while True:
    try:
      llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)
      chain3= final_summary_template | llm
      final_response=chain3.invoke({"community_summaries_text":community_summary})
      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED------------------------&&&&&&^^^^^^^%%%%%%%%%%%-----------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  pattern = r"<output>(.*?)</output>"
  match = re.search(pattern, final_response.content, re.DOTALL)
  if match:
    print(match.group(1).strip())
    df.at[current_abstract, 'summary']=match.group(1).strip()
  else:
    print(final_response.content)
    df.at[current_abstract, 'summary']=final_response.content
  df.to_excel(file_path,index=False)
  current_abstract+=1
  print("UPDATED SUCESSFULLY------------------------$$$$$$$$$$$$$$$%%%%%%%%%%%%%%%%%%%%-----------------------------")
